In [ ]:
# CloudCost IQ — Colab training (single-cell, end-to-end)
# Default: clones repo + runs SYNTHETIC DEMO training (not real billing / not for production accuracy claims).
# To use real data: set CLOUDCOST_DATA_SOURCE='drive' or 'upload' and provide a normalized billing Parquet.

import os, sys, platform, random, subprocess, json, shutil, importlib
from pathlib import Path

# ----------------------------
# Configuration (edit if needed; env vars override)
# ----------------------------
REPO_URL = os.environ.get('CLOUDCOST_REPO_URL', 'https://github.com/shadil-rayyan/CloudCost-IQ.git')
REPO_REF = os.environ.get('CLOUDCOST_REPO_REF', 'main')  # branch/tag/commit
PROJECT_ROOT = Path(os.environ.get('CLOUDCOST_PROJECT_ROOT', '/content/CloudCost-IQ')).resolve()

DATA_SOURCE = os.environ.get('CLOUDCOST_DATA_SOURCE', 'synthetic_demo')  # drive | upload | synthetic_demo
CURRENCY = os.environ.get('CLOUDCOST_CURRENCY', 'USD').upper()
COST_BASIS = os.environ.get('CLOUDCOST_COST_BASIS', 'billed')
DATA_PATH_ENV = os.environ.get('CLOUDCOST_DATA_PATH', '/content/drive/MyDrive/cloudcost-iq/normalized_billing.parquet')

COPY_TO_DRIVE = os.environ.get('CLOUDCOST_COPY_TO_DRIVE', '1').lower() in ('1', 'true', 'yes')
DRIVE_MODELS_DIR = os.environ.get('CLOUDCOST_DRIVE_MODELS_DIR', '/content/drive/MyDrive/cloudcost-iq/models')
TRACK_MLFLOW = os.environ.get('CLOUDCOST_MLFLOW', '0').lower() in ('1', 'true', 'yes')
SEED = int(os.environ.get('CLOUDCOST_SEED', '42'))

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print('=== Environment ===')
print('Python:', platform.python_version(), '|', platform.platform())
print('Config:', {'repo': REPO_URL, 'ref': REPO_REF, 'data_source': DATA_SOURCE, 'currency': CURRENCY, 'cost_basis': COST_BASIS})

# ----------------------------
# Install dependencies BEFORE importing heavy/binary packages (avoids ABI mismatches)
# ----------------------------
print('\n=== Installing dependencies ===')
base_pkgs = [
    'numpy>=1.26,<3', 'pandas>=2.2,<3', 'pyarrow>=15,<18',
    'scikit-learn>=1.4,<2', 'xgboost>=2.0,<3',
    'pydantic>=2.7,<3', 'pydantic-settings>=2.3,<3', 'PyYAML>=6',
    'plotly>=5.22,<6', 'mlflow>=2.12,<4', 'shap>=0.45,<1', 'joblib>=1.3,<2'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *base_pkgs])
importlib.invalidate_caches()

# ----------------------------
# Clone/load project
# ----------------------------
print('\n=== Project checkout ===')
if REPO_URL:
    if not PROJECT_ROOT.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all', '--tags'])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REF])
    try:
        commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
        print('Checked out commit:', commit)
    except Exception as e:
        print('Could not read git commit:', e)
else:
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd().resolve()
    print('Using existing checkout:', PROJECT_ROOT)

for req in (PROJECT_ROOT / 'requirements.txt', PROJECT_ROOT / 'requirements-ml.txt'):
    if req.exists():
        print('Installing', req.name)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
importlib.invalidate_caches()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

# Import scientific stack after installs
import numpy as np
import pandas as pd
try:
    import pyarrow as pa
    import pyarrow.parquet as pq  # health check
    import xgboost
    import sklearn
    print('Versions:', {'numpy': np.__version__, 'pandas': pd.__version__, 'pyarrow': pa.__version__, 'xgboost': xgboost.__version__, 'sklearn': sklearn.__version__})
except Exception as e:
    raise RuntimeError(
        'Binary package import failed (often due to Colab ABI/runtime state). '
        'Fix: Runtime -> Restart session/runtime, then run this single cell again from the top. '
        f'Original error: {type(e).__name__}: {e}'
    ) from e

np.random.seed(SEED)

# ----------------------------
# Load or generate dataset
# ----------------------------
print('\n=== Dataset ===')
RECORDS_PATH = None
if DATA_SOURCE == 'drive':
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    RECORDS_PATH = Path(DATA_PATH_ENV)
    if not RECORDS_PATH.exists():
        raise FileNotFoundError(f'CLOUDCOST_DATA_PATH not found: {RECORDS_PATH}. Provide a normalized billing Parquet file or directory.')
elif DATA_SOURCE == 'upload':
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    names = list(uploaded.keys())
    if not names:
        raise RuntimeError('No file uploaded.')
    RECORDS_PATH = Path('/content') / names[0]
    print('Uploaded:', RECORDS_PATH)
elif DATA_SOURCE == 'synthetic_demo':
    print('WARNING: generating SYNTHETIC DEMO billing data (not real). For pipeline demonstration only; not for production accuracy claims.')
    import scripts.create_demo_billing as demo_mod
    from app.utils.files import write_parquet_atomic
    demo_out = PROJECT_ROOT / 'data' / 'demo' / 'demo_synthetic_normalized.parquet'
    demo_out.parent.mkdir(parents=True, exist_ok=True)
    end = pd.Timestamp.today().normalize().date()
    frame_demo = demo_mod.build_frame(days=420, end=end, seed=SEED)
    write_parquet_atomic(frame_demo, demo_out)
    (demo_out.parent / 'demo_manifest.json').write_text(json.dumps({
        'synthetic': True,
        'warning': 'SYNTHETIC DEMONSTRATION DATA - not real billing, not for production accuracy claims.',
        'rows': int(len(frame_demo)), 'currency': CURRENCY, 'cost_basis': COST_BASIS, 'seed': SEED
    }, indent=2), encoding='utf-8')
    RECORDS_PATH = demo_out
    print('Synthetic demo data written to:', RECORDS_PATH)
else:
    raise ValueError("DATA_SOURCE must be 'drive', 'upload', or 'synthetic_demo'.")

print('Records path:', RECORDS_PATH, '| cohort:', CURRENCY, COST_BASIS)

# ----------------------------
# Prepare panel, features, splits
# ----------------------------
print('\n=== Prepare training data ===')
from app.ml.pipeline import load_normalized_records, prepare_training_panel, build_supervised, make_splits

records = load_normalized_records(RECORDS_PATH)
print('Normalized rows:', f'{len(records):,}')
if len(records) == 0:
    raise RuntimeError('No records loaded. Check CLOUDCOST_DATA_PATH / data source.')
if 'raw_fields_json' in records.columns:
    sample_raw = ' '.join(records['raw_fields_json'].dropna().astype(str).head(3).tolist()).lower()
    if 'synthetic' in sample_raw:
        print('*** SYNTHETIC/DEMO DATA DETECTED — demo only, not for production accuracy claims. ***')

prepared = prepare_training_panel(records, currency=CURRENCY, cost_basis=COST_BASIS)
quality = prepared.quality
print('Dataset quality (key fields):')
print(json.dumps({k: quality.get(k) for k in (
    'currency','cost_basis','series_count','eligible_series_count','history_start','history_end','history_days',
    'eligible_records','observed_records','missing_periods','zero_cost_days','sufficient','reason','dataset_fingerprint'
)}, indent=2, default=str))
if not prepared.is_sufficient:
    raise RuntimeError('Dataset is not eligible for training: ' + (prepared.insufficient_reason() or 'see dataset quality above'))

panel = prepared.eligible_panel if not prepared.eligible_panel.empty else prepared.panel
print('Panel rows:', len(panel), '| series:', panel['series_id'].nunique() if 'series_id' in panel.columns else 'n/a')

supervised = build_supervised(panel)
print('Supervised rows:', len(supervised.frame), '| features:', len(supervised.schema.all_features))
if supervised.frame.empty:
    raise RuntimeError('No supervised training rows could be built.')

splits = make_splits(supervised)
for msg in splits.messages:
    print('-', msg)
if not splits.is_valid:
    raise RuntimeError('Chronological split plan is incomplete (need rolling-origin folds and untouched final test).')
print('Valid split plan:', splits.is_valid, '| folds:', len(splits.folds), '| final test:', bool(splits.final_test))

# ----------------------------
# Train, evaluate, explain
# ----------------------------
print('\n=== Train & evaluate ===')
from app.ml.train import TrainingConfig, train_forecast_model
from app.ml.artifact_manager import ArtifactManager
from app.ml.model_contract import validate_bundle

config = TrainingConfig.from_ml_config(currency=CURRENCY, cost_basis=COST_BASIS)
eval_result = train_forecast_model(
    supervised, splits, dataset_quality=quality, config=config,
    save_bundle=False, track_mlflow=False,
)
print('Selected rounds:', eval_result.rounds, '| passes gates:', eval_result.passes_gates)
for gate in eval_result.gates:
    print(f"[{'PASS' if gate['passed'] else 'FAIL'}] {gate['name']}: {gate['detail']}")
print('Final test metrics:')
print(json.dumps(eval_result.metrics.get('final_test', {}), indent=2, default=str))
print('Baseline comparison:')
print(json.dumps(eval_result.metrics.get('baseline_comparison', {}), indent=2, default=str))

# Explainability (best-effort)
print('\n=== Explainability (best-effort) ===')
try:
    features = supervised.features()
    try:
        import shap
        sample = features.head(min(300, len(features)))
        explainer = shap.TreeExplainer(eval_result.model)
        values = explainer.shap_values(sample)
        matrix = np.asarray(values)
        if matrix.ndim == 3:
            matrix = matrix[:, :, 1] if matrix.shape[2] > 1 else matrix[:, :, 0]
        imp = (pd.DataFrame({'feature': sample.columns, 'mean_abs_shap': np.abs(matrix).mean(axis=0)})
               .sort_values('mean_abs_shap', ascending=False).head(15))
        print('Top features by mean |SHAP|:')
        print(imp.to_string(index=False))
    except Exception as e:
        print('SHAP unavailable:', type(e).__name__, e, '- falling back to gain importance')
        booster = eval_result.model.get_booster()
        gain = booster.get_score(importance_type='gain')
        imp = pd.DataFrame(sorted(gain.items(), key=lambda x: x[1], reverse=True)[:15], columns=['feature','gain'])
        print(imp.to_string(index=False))
except Exception as e:
    print('Explainability skipped:', type(e).__name__, e)
print('Note: feature importance indicates model usage, not causation.')

# ----------------------------
# Save candidate bundle, validate, package
# ----------------------------
print('\n=== Save & validate candidate bundle ===')
final_version = os.environ.get('CLOUDCOST_MODEL_VERSION') or ArtifactManager.new_version()
final_result = train_forecast_model(
    supervised, splits, dataset_quality=quality, config=config,
    version=final_version, save_bundle=True, track_mlflow=TRACK_MLFLOW,
)
print('Candidate version:', final_result.version)
print('Bundle path:', final_result.bundle_path)
print('Passes gates:', final_result.passes_gates)
for msg in final_result.messages:
    print('-', msg)

validation = validate_bundle(final_result.bundle_path)
for check in validation.checks:
    print(f"[{'PASS' if check.passed else 'FAIL'}] {check.name}: {check.detail}")
for err in validation.errors:
    print('ERROR:', err)
print('Bundle validation passed:', validation.passed)

bundle_dir = Path(final_result.bundle_path)
print('\nBundle contents:')
for p in sorted(bundle_dir.iterdir()):
    size = p.stat().st_size if p.is_file() else 0
    print(f' - {p.name:35s} {size:>10,} bytes')

# Zip for download
zip_path = Path('/content') / f'{bundle_dir.name}.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir.parent, bundle_dir.name)
print('\nZip ready:', zip_path)

# Optional copy to Drive
if COPY_TO_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
        dest_root = Path(DRIVE_MODELS_DIR)
        dest_root.mkdir(parents=True, exist_ok=True)
        dest = dest_root / bundle_dir.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(bundle_dir, dest)
        print('Copied bundle to Drive:', dest)
    except Exception as e:
        print('Drive copy skipped/failed:', type(e).__name__, e)

print('\n=== Next steps ===')
print('1) Download/copy the bundle folder to your local repo under artifacts/models/candidates/<version>')
print('2) Validate: python scripts/validate_model.py --version', bundle_dir.name)
if validation.passed and final_result.passes_gates:
    print('3) Promote (explicit): python scripts/validate_model.py --version', bundle_dir.name, '--promote')
else:
    print('3) Do NOT promote: validation/gates did not fully pass.')
if DATA_SOURCE == 'synthetic_demo' or 'synthetic' in json.dumps(quality, default=str).lower():
    print('\nREMINDER: This run used SYNTHETIC DEMO data — for pipeline demonstration only, not for production accuracy claims.')

print('\nDone.')
